# Microsoft 365 Usage — Audit Ingester (Fabric)

Calls the Microsoft Graph audit log for the Microsoft 365 activity the **M365 Usage Analytics** template reads
(file, mail, Teams and meeting operations, including **MailItemsAccessed** item counts, plus AI-app interactions) and
rolls each user's day up with the same rules as the template's own processor. All parsing runs in Spark.

`CopilotInteraction` is **not** fetched here: `Copilot_Audit_Log_Direct_Ingester` already writes it to
`dbo.Copilot_Interactions_Parsed`, and the template reads it from there.

**Output**
- `dbo.m365_usage_rollup` — one row per user, day, operation, workload, file type, app and agent (the template's *Rollup* file, without AI operations)
- `dbo.connected_ai_app_interactions` — one row per `ConnectedAIAppInteraction` audit record
- `dbo.ai_app_interactions` — one row per `AIAppInteraction` audit record

The template combines these with `Copilot_Interactions_Parsed`, `copilot_org_data` and `copilot_licensed_users` and
computes its rollup, session stats, per-user statistics, cohorts and user directory with T-SQL on the SQL analytics endpoint.

**Volume**: `MailItemsAccessed` is the largest operation by far on most tenants. Only rolled-up counts are kept, but the
fetch still reads every record, so start with a short backfill on a large tenant.

**Permissions**: `AuditLogsQuery.Read.All` (application, admin-consented). Mail and meeting events need Purview Audit enabled.

## 1. Configuration

Fill in the config values below. The client secret should ideally come from Key Vault in production.

**Lakehouse Schemas note**: the output tables use the `dbo.` prefix, which works for both schema-enabled and non-schema Lakehouses.

In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra -> Overview -> Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra -> App registrations -> your app -> Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault - see below.

# --- Load mode ---------------------------------------------------------------------
# 'backfill'    = pull BACKFILL_DAYS of history once (overwrite). Use for the initial load.
# 'incremental' = requery whole days from the trailing LOOKBACK_DAYS (late arrivals) and replace them.
MODE          = 'incremental'
BACKFILL_DAYS = 30                                    # history to pull when MODE='backfill' (up to 180)
LOOKBACK_DAYS = 7                                     # trailing requery window + empty-table fallback
OUTPUT_TABLE  = 'dbo.m365_usage_rollup'              # template 'Rollup' input
CONNECTED_AI_TABLE = 'dbo.connected_ai_app_interactions'  # ConnectedAIAppInteraction records
AI_APP_TABLE  = 'dbo.ai_app_interactions'           # AIAppInteraction records
OPERATIONS    = [                                    # audit operations the template reads
    'FileAccessed',
    'FileViewed',
    'FilePreviewed',
    'FileModified',
    'FileDownloaded',
    'FileUploaded',
    'MailItemsAccessed',
    'MailboxLogin',
    'Send',
    'MessageSent',
    'MessageRead',
    'MessagesListed',
    'ChatRetrieved',
    'ChatCreated',
    'MeetingParticipantJoined',
    'MeetingStarted',
    'MeetingEnded',
    'MeetingParticipantDetail',
    'MeetingDetail',
    'TeamsSessionStarted',
    'ConnectedAIAppInteraction',
    'AIAppInteraction',
]

# --- Backfill scale tuning ---------------------------------------------------------
CHUNK_HOURS             = 8     # window size; backfill = many windows. Use 2 on a very large tenant: 8h can fail in the audit service.
MAX_CONCURRENT_QUERIES  = 5     # Graph runs ~10 audit queries at once per tenant; 5 leaves room for other admins.
MAX_WAIT_MIN_PER_QUERY  = 240   # ceiling PER window (not the whole run). Resumable across reruns.
POLL_INTERVAL_SEC       = 30    # poll cadence per window
STAGING_DIR             = 'Files/_m365_audit_staging_raw'  # raw Graph records; bump the name if the staged format changes  # per-window JSONL + manifest; active runs only read their own windows.

# --- Failed-window recovery ----------------------------------------------------------
# A window whose query ends 'failed' or 'cancelled' (or times out) is sent again as a NEW query after a
# backoff. If it keeps failing it is split in half (8h -> 4h -> 2h -> 1h) and the halves are queried.
# Every window is attempted; the run only fails (without writing) if a window fails at MIN_CHUNK_HOURS.
WINDOW_RETRIES          = 3     # extra attempts per window before it is split
MIN_CHUNK_HOURS         = 1     # smallest size a failing window is split down to
SPLIT_ON_TIMEOUT        = True  # a window that hits MAX_WAIT_MIN_PER_QUERY splits at once instead of re-waiting
RETRY_BASE_SEC          = 60    # backoff before a retry: ~60s, 120s, 240s (with jitter) ...
RETRY_MAX_SEC           = 900   # ... never longer than this
MIN_CONCURRENT_QUERIES  = 1     # failures and HTTP 429s lower concurrency for the rest of the run, not below this
ALLOW_UNRECOVERED_GAPS  = False # incremental refuses to run while an older window failed and was never recovered

# --- Long runs (large tenants) -------------------------------------------------------
# With a budget, the run stops starting windows once it is used up and fails before anything is written,
# so the pipeline's retry carries on: finished windows are reused and queries still running in the audit
# service are picked up again rather than sent afresh. Keep it below the activity timeout, leaving time
# to read, parse and write what was fetched.
TIME_BUDGET_MIN         = 0     # minutes for fetching windows; 0 = no budget
RESUME_QUERY_HOURS      = 12    # a query an earlier attempt left running is re-polled if it is this recent
TRAILING_REUSE_HOURS    = 6     # a trailing window fetched this recently is reused, not queried again

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'M365UsageAppSecret')

## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry

# Resilient HTTP session. The Microsoft Graph security/auditLog endpoints intermittently return
# 502/503/504 (gateway timeouts) and 429 (throttling), especially during the poll + records-fetch
# phases on busy tenants. A shared Session with urllib3 Retry transparently retries those with
# exponential backoff + Retry-After, so a transient blip no longer aborts the whole run.
THROTTLE_LISTENERS = []  # step 5 registers its query limiter here; called on every HTTP 429.


def _notify_throttle():
    for listener in list(THROTTLE_LISTENERS):
        try:
            listener()
        except Exception:
            pass


class _ThrottleAwareRetry(Retry):
    # urllib3 retries 429s before _request sees them; report each one so concurrency can back off.
    def increment(self, method=None, url=None, response=None, *args, **kwargs):
        if response is not None and getattr(response, 'status', None) == 429:
            _notify_throttle()
        return super().increment(method, url, response, *args, **kwargs)


def _build_session() -> requests.Session:
    s = requests.Session()
    retry = _ThrottleAwareRetry(
        total=8,                       # up to 8 attempts per request
        connect=5, read=5, status=8,
        backoff_factor=2,              # 0,2,4,8,16,32... seconds (capped by urllib3)
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = _build_session()

# Belt-and-braces wrapper: urllib3 Retry handles most cases, but we add an outer retry loop so a
# request that *still* surfaces a transient status (or a connection error) after the adapter gives
# up is retried a few more times rather than killing the notebook.
_TRANSIENT = {429, 500, 502, 503, 504}

def _request(method: str, url: str, *, attempts: int = 5, **kwargs):
    kwargs.setdefault('timeout', 60)
    last = None
    for i in range(1, attempts + 1):
        try:
            r = SESSION.request(method, url, **kwargs)
        except requests.exceptions.RequestException as e:
            last = e
            if i == attempts:
                raise
            wait = min(60, 2 ** i)
            print(f'    transient network error ({type(e).__name__}); retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        if r.status_code == 429:
            _notify_throttle()
        if r.status_code in _TRANSIENT and i < attempts:
            wait = min(60, 2 ** i)
            print(f'    transient HTTP {r.status_code} on {method} {url.split("?")[0].rsplit("/",1)[-1]}; '
                  f'retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        return r
    return r  # pragma: no cover

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = _request('POST', url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

get_headers()
print('\u2713 Graph token acquired (auto-refreshing, retry-resilient session).')


## 3. Build query windows + the `create_query` helper

The lookback is split into smaller time windows (`CHUNK_HOURS`). Each window is its own async
audit query, which completes far faster than one giant query — the main cause of timeouts on
large tenants — and lets the run checkpoint window-by-window.

In [ ]:
from datetime import datetime, timezone, timedelta
import hashlib, json, random, threading, time

WINDOW_KEY_VERSION = 2
KEY_COLUMNS = ['Id', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey']
_RESOURCE_KEY_FIELDS = ('Type', 'Action', 'SiteUrl', 'SensitivityLabelId')
_SYNTHETIC_AUDIT_FIELDS = frozenset({'_StableKey', '_StableOrdinal'})


def _as_utc_datetime(value):
    if value in (None, ''):
        return None
    if isinstance(value, datetime):
        dt = value
    else:
        text = str(value).strip()
        if not text:
            return None
        if text.endswith('Z'):
            text = text[:-1] + '+00:00'
        dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _norm_text(value) -> str:
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def _coerce_json_value(value):
    if value is None:
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text
    return value


def _has_meaningful_value(value) -> bool:
    if isinstance(value, dict):
        return any(_has_meaningful_value(v) for k, v in value.items() if k not in _SYNTHETIC_AUDIT_FIELDS)
    if isinstance(value, list):
        return any(_has_meaningful_value(v) for v in value)
    if value is None:
        return False
    if isinstance(value, str):
        return _norm_text(value) != ''
    return True


def _canonical_json_text(value) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))


def _stable_message_base_key(message) -> str:
    message = dict(message or {})
    msg_id = _norm_text(message.get('Id'))
    if msg_id:
        return f'mid:{msg_id}'
    payload = {k: v for k, v in message.items() if k not in _SYNTHETIC_AUDIT_FIELDS and k != 'Id'}
    return 'msgf:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_resource_base_key(resource) -> str:
    resource = dict(resource or {})
    payload = {k: v for k, v in resource.items() if k not in _SYNTHETIC_AUDIT_FIELDS}
    if not _has_meaningful_value(payload):
        return 'resource:none'
    return 'resource:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_sequence(items, base_key_builder, path) -> list:
    prepared = []
    for item in items or []:
        canonical_item = _canonicalize_json(item, path)
        if not isinstance(canonical_item, dict):
            canonical_item = {'Value': canonical_item}
        canonical_text = _canonical_json_text(canonical_item)
        base_key = base_key_builder(canonical_item)
        prepared.append((base_key, canonical_text, canonical_item))
    prepared.sort(key=lambda entry: (entry[0], entry[1]))
    totals = {}
    for base_key, _canonical_text, _item in prepared:
        totals[base_key] = totals.get(base_key, 0) + 1
    seen = {}
    stable = []
    for ordinal, (base_key, _canonical_text, canonical_item) in enumerate(prepared):
        seen[base_key] = seen.get(base_key, 0) + 1
        stable_key = base_key if totals[base_key] == 1 else f'{base_key}#{seen[base_key]}'
        enriched = dict(canonical_item)
        enriched['_StableKey'] = stable_key
        enriched['_StableOrdinal'] = ordinal
        stable.append(enriched)
    return stable


def _canonicalize_json(value, path=()):
    value = _coerce_json_value(value) if not path else value
    if isinstance(value, dict):
        return {
            str(key): _canonicalize_json(val, path + (str(key),))
            for key, val in sorted(value.items(), key=lambda item: str(item[0]))
            if key not in _SYNTHETIC_AUDIT_FIELDS
        }
    if isinstance(value, list):
        if path == ('CopilotEventData', 'Messages'):
            return _stable_sequence(value, _stable_message_base_key, path)
        if path == ('CopilotEventData', 'AccessedResources'):
            return _stable_sequence(value, _stable_resource_base_key, path)
        return [_canonicalize_json(item, path) for item in value]
    return value


def canonicalize_audit_payload(audit_data):
    value = _coerce_json_value(audit_data)
    if value is None:
        return None
    return _canonicalize_json(value)


def determine_incremental_start(end_dt, high_water_mark, lookback_days: int):
    lookback_days = max(int(lookback_days), 0)
    lookback_floor = end_dt - timedelta(days=lookback_days)
    hw = _as_utc_datetime(high_water_mark)
    if hw is None:
        return lookback_floor
    return min(hw, lookback_floor)


def validate_incremental_target_columns(columns, required_columns, table_name: str):
    cols = set(columns or [])
    missing = [c for c in required_columns if c not in cols]
    if missing:
        raise RuntimeError(
            f"Legacy table {table_name} is missing stable key columns {missing}. "
            "Run one MODE='backfill' overwrite to upgrade the parsed audit table before incremental resumes."
        )


def build_source_record_key(
    record_id,
    creation_date=None,
    operation=None,
    audit_data=None,
    record_type=None,
    associated_admin_units=None,
    associated_admin_units_names=None,
) -> str:
    rid = _norm_text(record_id)
    if rid:
        return f"rid:{rid}"
    payload = {
        'CreationDate': _norm_text(creation_date),
        'Operation': _norm_text(operation),
        'RecordType': _norm_text(record_type),
        'AuditData': canonicalize_audit_payload(audit_data),
        'AssociatedAdminUnits': _canonicalize_json(associated_admin_units or []),
        'AssociatedAdminUnitsNames': _canonicalize_json(associated_admin_units_names or []),
    }
    return 'synthetic:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def canonicalize_audit_record(record) -> dict:
    record = record or {}
    audit_payload = canonicalize_audit_payload(record.get('auditData'))
    return {
        'RecordId': record.get('id'),
        'CreationDate': record.get('createdDateTime'),
        'RecordType': record.get('auditLogRecordType'),
        'Operation': record.get('operation'),
        'AuditData': _canonical_json_text(audit_payload) if audit_payload is not None else None,
        'SourceRecordKey': build_source_record_key(
            record.get('id'),
            record.get('createdDateTime'),
            record.get('operation'),
            audit_payload,
            record_type=record.get('auditLogRecordType'),
            associated_admin_units=record.get('associatedAdminUnits', []),
            associated_admin_units_names=record.get('associatedAdminUnitsNames', []),
        ),
        'AssociatedAdminUnits': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnits', []))),
        'AssociatedAdminUnitsNames': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnitsNames', []))),
    }


def stable_window_key(win_start, win_end) -> str:
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    return f"v{WINDOW_KEY_VERSION}_{ws:%Y%m%d%H%M}_{we:%Y%m%d%H%M}"


# --- Failed-window recovery: keep in step with the recovery logic in the processor -----------
# A split window keeps its own stable_window_key; its halves get theirs, so manifests written before
# splitting existed stay valid and every part is reusable on a rerun.
def split_window(win_start, win_end, min_hours):
    """Halves of a window for another try at a smaller size; [] when a half would be under min_hours."""
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    half_minutes = int((we - ws).total_seconds() // 60) // 2
    if half_minutes <= 0 or half_minutes < float(min_hours) * 60:
        return []
    mid = win_start + timedelta(minutes=half_minutes)
    return [(win_start, mid), (mid, win_end)]


def expand_split_windows(manifest, win_start, win_end, min_hours, include_parents=False):
    """The windows to query for one grid window: its halves (recursively) if an earlier run split it."""
    entry = (manifest or {}).get(stable_window_key(win_start, win_end))
    halves = []
    if isinstance(entry, dict) and entry.get('status') == 'split':
        halves = split_window(win_start, win_end, min_hours)
    if not halves:
        return [(win_start, win_end)]
    out = [(win_start, win_end)] if include_parents else []
    for ws, we in halves:
        out.extend(expand_split_windows(manifest, ws, we, min_hours, include_parents))
    return out


def retry_delay(attempt, base_seconds, max_seconds, rand=random.random):
    """Exponential backoff with jitter: base, 2x base, 4x base ... capped at max_seconds, times 50-100%."""
    raw = min(float(max_seconds), float(base_seconds) * (2 ** max(int(attempt) - 1, 0)))
    return max(0.0, raw * (0.5 + 0.5 * rand()))


class AdaptiveLimiter:
    """Caps concurrent audit queries. Failures and throttling lower the cap for the rest of the run.

    A burst of failures with one cause (several windows failing in the same second) counts once:
    the cap moves at most once per cooldown_seconds.
    """

    def __init__(self, limit, minimum=1, cooldown_seconds=60, clock=time.monotonic):
        self.minimum = max(1, int(minimum))
        self.limit = max(self.minimum, int(limit))
        self.active = 0
        self.history = []
        self._cooldown = float(cooldown_seconds)
        self._clock = clock
        self._last_shrink = None
        self._cond = threading.Condition()

    def __enter__(self):
        with self._cond:
            while self.active >= self.limit:
                self._cond.wait()
            self.active += 1
        return self

    def __exit__(self, *exc):
        with self._cond:
            self.active -= 1
            self._cond.notify_all()
        return False

    def shrink(self, reason, halve=False):
        with self._cond:
            now = self._clock()
            if self._last_shrink is not None and now - self._last_shrink < self._cooldown:
                return False
            new = max(self.minimum, self.limit // 2 if halve else self.limit - 1)
            if new >= self.limit:
                return False
            self.history.append((self.limit, new, reason))
            self.limit, self._last_shrink = new, now
            return True


def _manifest_range(entry):
    if not isinstance(entry, dict):
        return None
    try:
        ws = _as_utc_datetime(entry.get('window_start'))
        we = _as_utc_datetime(entry.get('window_end'))
    except (TypeError, ValueError):
        return None
    return (ws, we) if ws and we and ws < we else None


def uncovered_failed_ranges(manifest, not_before, before):
    """Time ranges in [not_before, before) of unfinished windows that no succeeded window covers."""
    lo = _as_utc_datetime(not_before)
    hi = _as_utc_datetime(before)
    entries = [e for e in (manifest or {}).values() if isinstance(e, dict)]
    covered = sorted(r for r in (_manifest_range(e) for e in entries if e.get('status') == 'succeeded') if r)
    gaps = []
    for entry in entries:
        if entry.get('status') in ('succeeded', 'split'):
            continue
        rng = _manifest_range(entry)
        if not rng:
            continue
        pieces = [(max(rng[0], lo), min(rng[1], hi))]
        for cs, ce in covered:
            pieces = [p for s, e in pieces for p in ((s, min(e, cs)), (max(s, ce), e)) if p[0] < p[1]]
        gaps.extend(pieces)
    merged = []
    for s, e in sorted(gaps):
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


if MODE not in ('backfill', 'incremental'):
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'backfill' or 'incremental'.")

end_date = datetime.now(timezone.utc)
if MODE == 'backfill':
    start_date = end_date - timedelta(days=BACKFILL_DAYS)
    WRITE_MODE = 'overwrite'
else:
    WRITE_MODE = 'replace_dates'
    _hw = None
    if spark.catalog.tableExists(OUTPUT_TABLE):
        from pyspark.sql import functions as F
        _hw = spark.table(OUTPUT_TABLE).agg(F.max('CreationDate').alias('m')).collect()[0]['m']
        _hw = datetime.strptime(_hw[:10], '%Y-%m-%d').replace(tzinfo=timezone.utc) if _hw else None
    # Rollups are per day, so always start at midnight UTC and replace whole days.
    start_date = determine_incremental_start(end_date, _hw, LOOKBACK_DAYS).replace(
        hour=0, minute=0, second=0, microsecond=0)
    if _hw:
        print(
            f'incremental: high-water mark = {_as_utc_datetime(_hw):%Y-%m-%d %H:%M}; '
            f'requerying trailing {LOOKBACK_DAYS}d from {start_date:%Y-%m-%d %H:%M}'
        )
    else:
        print(f'incremental: no prior data, defaulting to last {LOOKBACK_DAYS}d')

_anchor_hour = start_date.hour - (start_date.hour % CHUNK_HOURS)
_cur = start_date.replace(hour=_anchor_hour, minute=0, second=0, microsecond=0)
windows = []
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
REPLACE_FROM = f'{start_date:%Y-%m-%d}T00:00:00.000Z'
print(f'MODE={MODE}: {start_date:%Y-%m-%d} -> {end_date:%Y-%m-%d}  =>  {len(windows)} window(s) of {CHUNK_HOURS}h, WRITE_MODE={WRITE_MODE}')


def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'M365 Usage {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   [],
        'operationFilters':    OPERATIONS,
    }
    r = _request('POST', 'https://graph.microsoft.com/beta/security/auditLog/queries',
                 json=body, headers=get_headers(), timeout=30)
    r.raise_for_status()
    return r.json()['id']

## 4. Resumable wait helper

Polls an async query until it succeeds, with a **generous, configurable** ceiling
(`MAX_WAIT_MIN_PER_QUERY`) instead of a hard 30-minute cut-off. The token auto-refreshes while
waiting, so multi-hour jobs don't fail on expiry.

A query the service ends as `failed` or `cancelled` raises `AuditQueryFailed`. Step 5 retries it with a
fresh query, and splits the window if it keeps failing.

In [ ]:
import time


class AuditQueryFailed(RuntimeError):
    """The audit service ended a query as failed or cancelled; a new query for the window may succeed."""


class BudgetReached(RuntimeError):
    """TIME_BUDGET_MIN ran out. The window is left for the retry, which re-polls its query."""


def budget_exhausted(deadline, now=None) -> bool:
    return deadline is not None and (time.monotonic() if now is None else now) >= deadline


RUN_DEADLINE = time.monotonic() + TIME_BUDGET_MIN * 60 if TIME_BUDGET_MIN and TIME_BUDGET_MIN > 0 else None


def _read_query_status(query_id: str):
    r = _request(
        'GET',
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
        headers=get_headers(),
        timeout=30,
    )
    r.raise_for_status()
    payload = r.json()
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {query_id} returned malformed status payload type: {type(payload).__name__}'
        )
    status = payload.get('status')
    if not isinstance(status, str) or not status.strip():
        raise RuntimeError(f'Query {query_id} returned missing/invalid status: {status!r}')
    return payload, status.strip().lower()


def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY):
    started = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        if budget_exhausted(RUN_DEADLINE):
            raise BudgetReached(f'Time budget reached while query {query_id} was still running')
        polls += 1
        payload, status = _read_query_status(query_id)
        if status == 'succeeded':
            return payload
        if status in ('failed', 'cancelled'):
            raise AuditQueryFailed(f'Query {query_id} ended with status: {status}')
        if status not in ('notstarted', 'running'):
            raise RuntimeError(f'Query {query_id} returned unexpected status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.'
    )


## 5. Fetch records → stream to Lakehouse Files (chunked, bounded memory)

Each page of results is written straight to Lakehouse **Files** as JSON-lines, so the full
result set is **never** held in driver memory — the other main cause of OOM/timeouts on large
tenants. Set `WRITE_MODE = 'overwrite'` to clear the staging area first.

**Failed windows recover on their own.** On large tenants a query sometimes ends `failed` or `cancelled`,
often when several queries are submitted at once or someone else is running audit searches. Each failing
window is:

1. sent again as a **new** query, up to `WINDOW_RETRIES` times, with exponential backoff and jitter;
2. then **split in half** (8h → 4h → 2h → 1h, down to `MIN_CHUNK_HOURS`), and the halves are queried.
   The halves are recorded in the manifest, so a rerun reuses every half that already succeeded.

Each failure or HTTP 429 also lowers the number of concurrent queries for the rest of the run
(not below `MIN_CONCURRENT_QUERIES`). Every window is attempted before the run reports anything.
If a window still fails at `MIN_CHUNK_HOURS`, the cell stops with one message listing the failed windows
and **nothing is written**. Rerun the notebook (in `MODE='backfill'` if this was a backfill); succeeded
windows are reused, so only the failed ones are queried again. An incremental run refuses to start while
an older window is unrecovered, so switching modes can't silently leave a gap.

In [ ]:
import json, os, tempfile, threading, time
from urllib.parse import urlparse
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait as wait_futures

# Defaults for config cells copied from an older version of this notebook.
WINDOW_RETRIES = int(globals().get('WINDOW_RETRIES', 3))
MIN_CHUNK_HOURS = float(globals().get('MIN_CHUNK_HOURS', 1))
SPLIT_ON_TIMEOUT = bool(globals().get('SPLIT_ON_TIMEOUT', True))
RETRY_BASE_SEC = float(globals().get('RETRY_BASE_SEC', 60))
RETRY_MAX_SEC = float(globals().get('RETRY_MAX_SEC', 900))
MIN_CONCURRENT_QUERIES = int(globals().get('MIN_CONCURRENT_QUERIES', 1))
ALLOW_UNRECOVERED_GAPS = bool(globals().get('ALLOW_UNRECOVERED_GAPS', False))

# Resumable, bounded-concurrency backfill. Each window's records stream to a JSONL file and its
# status is recorded in a manifest; rerun skips older succeeded windows, but recent LOOKBACK_DAYS
# windows are intentionally re-queried for late arrivals. Active runs only read the current window set.
# A failing window is retried with a new query, then split into halves (status 'split', parts=[keys]);
# the halves have their own manifest entries and staging files.

def _normalize_stage_dir(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage directory must be a non-empty string; found {path!r}.')
    text = path.strip()
    if text.startswith('abfss://'):
        return text.rstrip('/')
    normalized = text.replace('\\', '/')
    mount_prefix = '/lakehouse/default/'
    if normalized.startswith(mount_prefix):
        normalized = normalized[len(mount_prefix):]
    if normalized == 'Files' or normalized.startswith('Files/'):
        return normalized.rstrip('/')
    return os.path.abspath(text)


def _is_remote_stage_dir(path: str) -> bool:
    return isinstance(path, str) and (path == 'Files' or path.startswith('Files/') or path.startswith('abfss://'))


def _stage_path_join(base_dir: str, name: str) -> str:
    if _is_remote_stage_dir(base_dir):
        return base_dir.rstrip('/') + '/' + name.lstrip('/')
    return os.path.join(base_dir, name)


def _stage_basename(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage path must be a non-empty string; found {path!r}.')
    return path.rstrip('/').replace('\\', '/').split('/')[-1]


def _file_uri(path: str) -> str:
    return 'file:' + os.path.abspath(path)


def _driver_temp_root(base_dir: str) -> str:
    configured = globals().get('LOCAL_STAGE_TMP_DIR')
    if isinstance(configured, str) and configured.strip():
        root = os.path.abspath(configured)
    elif _is_remote_stage_dir(base_dir):
        root = os.path.abspath(os.path.join(tempfile.gettempdir(), 'universal_lake_audit_stage'))
    else:
        root = os.path.abspath(base_dir)
    os.makedirs(root, exist_ok=True)
    return root


def _new_local_temp_path(base_dir: str, stem: str, suffix: str) -> str:
    fd, temp_path = tempfile.mkstemp(
        prefix=f'.{stem}.{os.getpid()}.{threading.get_ident()}.',
        suffix=suffix,
        dir=_driver_temp_root(base_dir),
    )
    os.close(fd)
    return temp_path


def _get_notebook_fs():
    fs = globals().get('NOTEBOOK_FS')
    if fs is None:
        try:
            import notebookutils
        except ImportError as e:
            raise RuntimeError('Remote staging paths require notebookutils.fs in the Fabric runtime.') from e
        fs = getattr(notebookutils, 'fs', None)
    if fs is None:
        raise RuntimeError('Remote staging paths require notebookutils.fs.')
    missing = [name for name in ('ls', 'exists', 'rm', 'cp', 'mv', 'mkdirs') if not hasattr(fs, name)]
    if missing:
        raise RuntimeError(f'notebookutils.fs is missing required method(s): {missing}.')
    return fs


def _ensure_stage_dir(base_dir: str):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            fs.mkdirs(base_dir)
        return
    os.makedirs(base_dir, exist_ok=True)


def _list_stage_paths(base_dir: str) -> list:
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            return []
        paths = []
        for entry in fs.ls(base_dir):
            path = getattr(entry, 'path', None)
            if path is None and isinstance(entry, dict):
                path = entry.get('path')
            if not isinstance(path, str) or not path.strip():
                raise RuntimeError(f'Stage listing entry for {base_dir} is missing a usable path: {entry!r}')
            paths.append(path.rstrip('/'))
        return sorted(set(paths))
    if not os.path.isdir(base_dir):
        return []
    return sorted(os.path.join(base_dir, name) for name in os.listdir(base_dir))


def _stage_exists(path: str) -> bool:
    if _is_remote_stage_dir(path):
        return _get_notebook_fs().exists(path)
    return os.path.exists(path)


def _stage_remove(path: str):
    if _is_remote_stage_dir(path):
        fs = _get_notebook_fs()
        if fs.exists(path):
            if fs.rm(path, False) is False:
                raise RuntimeError(f'Failed to remove staged file {path}.')
        return
    if os.path.exists(path):
        os.remove(path)


def _write_stage_text(base_dir: str, target_path: str, text: str):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.tmp')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging file to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _write_stage_jsonl(base_dir: str, target_path: str, records):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.jsonl')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            for record in records:
                f.write(json.dumps(_row(record)) + '\n')
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging page to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _move_stage_file(base_dir: str, source_path: str, target_path: str, overwrite: bool = True):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        # Fabric's third argument is create_path, not overwrite. Rename in place
        # without first deleting the valid destination checkpoint.
        moved = fs.mv(source_path, target_path, False, overwrite)
        if moved is False:
            raise RuntimeError(f'Failed to move staged file {source_path} to {target_path}.')
        return
    if not overwrite and os.path.exists(target_path):
        raise FileExistsError(target_path)
    os.replace(source_path, target_path)


def _read_stage_json(base_dir: str, source_path: str):
    if _is_remote_stage_dir(base_dir):
        local_temp = _new_local_temp_path(base_dir, _stage_basename(source_path), '.json')
        try:
            copied = _get_notebook_fs().cp(source_path, _file_uri(local_temp))
            if copied is False:
                raise RuntimeError(f'Failed to copy staged file {source_path} to local scratch.')
            with open(local_temp, encoding='utf-8') as f:
                return json.load(f)
        finally:
            if os.path.exists(local_temp):
                os.remove(local_temp)
    with open(source_path, encoding='utf-8') as f:
        return json.load(f)


STAGING_ABS = _normalize_stage_dir(STAGING_DIR)
MANIFEST = _stage_path_join(STAGING_ABS, '_manifest.json')
MANIFEST_LOCK = threading.RLock()
_ensure_stage_dir(STAGING_ABS)


def list_window_files(base_dir: str, key: str, include_partial: bool = False):
    prefix = f'win_{key}_'
    matches = []
    for path in _list_stage_paths(base_dir):
        name = _stage_basename(path)
        if not name.startswith(prefix):
            continue
        if name.endswith('.jsonl') or (include_partial and name.endswith('.jsonl.partial')):
            matches.append(path)
    return sorted(set(matches))


def purge_window_files(base_dir: str, key: str, include_partial: bool = True) -> int:
    removed = 0
    for path in list_window_files(base_dir, key, include_partial=include_partial):
        if _stage_exists(path):
            _stage_remove(path)
            removed += 1
    return removed


def list_active_stage_files(base_dir: str, active_keys) -> list:
    files = []
    for key in sorted(set(active_keys or [])):
        files.extend(list_window_files(base_dir, key, include_partial=False))
    return sorted(files)


def _allowed_manifest_statuses():
    return {'querying', 'waiting', 'draining', 'succeeded', 'failed', 'split'}


def _validate_manifest_payload(data):
    if not isinstance(data, dict):
        raise RuntimeError(
            f'Manifest {MANIFEST} must be a JSON object keyed by window id; found {type(data).__name__}.'
        )
    allowed = _allowed_manifest_statuses()
    for key, entry in data.items():
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError(f'Manifest {MANIFEST} has an invalid window key: {key!r}.')
        if not isinstance(entry, dict):
            raise RuntimeError(
                f'Manifest {MANIFEST} entry {key!r} must be a JSON object; found {type(entry).__name__}.'
            )
        status = entry.get('status')
        if not isinstance(status, str) or status not in allowed:
            raise RuntimeError(
                f"Manifest {MANIFEST} entry {key!r} has invalid status {status!r}; "
                f'expected one of {sorted(allowed)}.'
            )
        if status == 'split':
            parts = entry.get('parts')
            if not isinstance(parts, list) or not parts or not all(isinstance(p, str) and p.strip() for p in parts):
                raise RuntimeError(f'Manifest {MANIFEST} entry {key!r} is split but has no valid parts list.')
    return data


def _load_manifest():
    if not _stage_exists(MANIFEST):
        return {}
    try:
        data = _read_stage_json(STAGING_ABS, MANIFEST)
    except json.JSONDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid JSON: {e}') from e
    except UnicodeDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid UTF-8 JSON: {e}') from e
    return _validate_manifest_payload(data)


def _manifest_temp_path():
    base = _stage_basename(MANIFEST)
    return _stage_path_join(STAGING_ABS, f'.{base}.{os.getpid()}.{threading.get_ident()}.tmp')


def _save_manifest(m):
    payload = _validate_manifest_payload(dict(m))
    _ensure_stage_dir(STAGING_ABS)
    temp_path = _manifest_temp_path()
    try:
        _write_stage_text(STAGING_ABS, temp_path, json.dumps(payload, indent=2, sort_keys=True))
        _move_stage_file(STAGING_ABS, temp_path, MANIFEST, overwrite=True)
    finally:
        if _stage_exists(temp_path):
            _stage_remove(temp_path)


def manifest_succeeded(entry) -> bool:
    return isinstance(entry, dict) and entry.get('status') == 'succeeded'


def should_refresh_window(win_end, manifest_entry, trailing_cutoff, fresh_after=None) -> bool:
    """Whether a window must be queried again. Trailing windows are, for late records, unless they
    were fetched after `fresh_after` (a retry soon after the run that fetched them)."""
    if not manifest_succeeded(manifest_entry):
        return True
    completed_at = _as_utc_datetime((manifest_entry or {}).get('completed_at'))
    if completed_at is None:
        return True
    if _as_utc_datetime(win_end) <= _as_utc_datetime(trailing_cutoff):
        return False
    return fresh_after is None or completed_at < _as_utc_datetime(fresh_after)


def trailing_fresh_after(now, reuse_hours):
    return now - timedelta(hours=reuse_hours) if reuse_hours and reuse_hours > 0 else None


def resumable_query_id(entry, now, max_hours):
    """The query an earlier attempt left running for this window, if it is recent enough to re-poll."""
    entry = entry or {}
    qid = entry.get('query_id')
    if entry.get('status') not in ('querying', 'waiting', 'draining') or not max_hours or max_hours <= 0:
        return None
    if not isinstance(qid, str) or not qid.strip():
        return None
    created = _as_utc_datetime(entry.get('query_created_at') or entry.get('refreshed_at'))
    if created is None or now - created > timedelta(hours=max_hours):
        return None
    return qid.strip()


def window_stage_files_reusable(base_dir: str, key: str, manifest_entry) -> bool:
    entry = manifest_entry or {}
    recorded = entry.get('files')
    if isinstance(recorded, list):
        if recorded:
            expected = [_stage_path_join(base_dir, _stage_basename(path)) for path in recorded]
            return all(_stage_exists(path) for path in expected)
        return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0
    files = list_window_files(base_dir, key, include_partial=False)
    if files:
        return True
    return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0


def is_window_reusable(base_dir: str, key: str, win_end, manifest_entry, trailing_cutoff, fresh_after=None) -> bool:
    return (not should_refresh_window(win_end, manifest_entry, trailing_cutoff, fresh_after)) and window_stage_files_reusable(
        base_dir, key, manifest_entry
    )


def _read_manifest_entry(key: str) -> dict:
    with MANIFEST_LOCK:
        return dict(manifest.get(key, {}))


def _mark_window(key: str, status: str, **extra):
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f'Window key must be a non-empty string; found {key!r}.')
    if not isinstance(status, str) or status not in _allowed_manifest_statuses():
        raise ValueError(f'Unsupported manifest status {status!r}.')
    with MANIFEST_LOCK:
        # The initial disk snapshot is loaded once. Remote listings or mounted reads can lag a
        # successful publish; reloading here can discard newer checkpoints from other workers.
        # Sharing a staging directory between separate runs remains unsupported.
        current = dict(manifest)
        entry = dict(current.get(key, {}))
        entry.update(extra)
        entry['status'] = status
        current[key] = entry
        _save_manifest(current)
        manifest.clear()
        manifest.update(current)


_STAGE_FIELDS = ('id', 'createdDateTime', 'operation', 'userPrincipalName', 'userId', 'auditLogRecordType', 'auditData')


def _row(r):
    """Stage only the fields the Spark parser reads (section 6); no per-record parsing here."""
    r = r or {}
    row = {k: r.get(k) for k in _STAGE_FIELDS}
    ad = row.get('auditData')
    if ad is not None and not isinstance(ad, str):
        row['auditData'] = json.dumps(ad, ensure_ascii=False, separators=(',', ':'))
    return row


def _is_graph_records_url(url: str, qid: str) -> bool:
    parsed = urlparse(url)
    return (
        parsed.scheme == 'https'
        and parsed.netloc == 'graph.microsoft.com'
        and parsed.path.startswith(f'/beta/security/auditLog/queries/{qid}/records')
    )


def _validate_graph_records_url(url, qid: str) -> str:
    if not isinstance(url, str) or not url.strip():
        raise RuntimeError(f'Query {qid} returned missing/invalid @odata.nextLink: {url!r}')
    url = url.strip()
    if not _is_graph_records_url(url, qid):
        raise RuntimeError(f'Query {qid} returned non-Graph records continuation URL: {url}')
    return url


def _validate_records_page(qid: str, current_url: str, payload):
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {qid} returned malformed records payload type {type(payload).__name__} for {current_url}'
        )
    if 'value' not in payload:
        raise RuntimeError(f'Query {qid} records payload missing value list for {current_url}')
    vals = payload['value']
    if not isinstance(vals, list):
        raise RuntimeError(f'Query {qid} records payload value must be a list for {current_url}')
    for index, record in enumerate(vals):
        if not isinstance(record, dict):
            raise RuntimeError(
                f'Query {qid} records payload item {index} must be an object for {current_url}'
            )
    next_link = payload.get('@odata.nextLink')
    if next_link is None:
        return vals, None
    return vals, _validate_graph_records_url(next_link, qid)


def _drain(qid, key):
    n, page = 0, 0
    partial_files = []
    purge_window_files(STAGING_ABS, key, include_partial=True)
    url = _validate_graph_records_url(
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{qid}/records?$top=999',
        qid,
    )
    seen_urls = set()
    while url:
        if url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {url}')
        seen_urls.add(url)
        r = _request('GET', url, headers=get_headers(), timeout=60)
        r.raise_for_status()
        vals, next_url = _validate_records_page(qid, url, r.json())
        if next_url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {next_url}')
        if vals:
            part_path = _stage_path_join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl.partial')
            _write_stage_jsonl(STAGING_ABS, part_path, vals)
            partial_files.append(part_path)
            page += 1
            n += len(vals)
        url = next_url
    for part_path in partial_files:
        _move_stage_file(STAGING_ABS, part_path, part_path[:-8], overwrite=True)
    return n, page


def _process(win, attempt=1):
    ws, we = win
    key = stable_window_key(ws, we)
    entry = _read_manifest_entry(key)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    now = datetime.now(timezone.utc)
    if is_window_reusable(STAGING_ABS, key, we, entry, trailing_cutoff, trailing_fresh_after(now, TRAILING_REUSE_HOURS)):
        return key, entry.get('rows', 0), 'skip'
    if budget_exhausted(RUN_DEADLINE):
        raise BudgetReached('Time budget reached before this window started')
    resume_qid = resumable_query_id(entry, now, RESUME_QUERY_HOURS) if attempt == 1 else None
    _mark_window(
        key,
        'querying',
        window_start=_as_utc_datetime(ws).isoformat(),
        window_end=_as_utc_datetime(we).isoformat(),
        refreshed_at=datetime.now(timezone.utc).isoformat(),
        attempts=attempt,
        error=None,
        **({'attempt_errors': []} if attempt == 1 else {}),
    )
    qid = None
    try:
        # Only creating and waiting take one of the tenant's concurrent query slots; draining doesn't.
        if resume_qid:
            qid = resume_qid
            try:
                with QUERY_LIMITER:
                    _mark_window(key, 'waiting', query_id=qid)
                    wait_for_query(qid)
                print(f'    {key}: picked up query {qid} from an earlier attempt')
            except BudgetReached:
                raise
            except Exception as exc:
                if _http_status(exc) in _FATAL_HTTP:
                    raise
                print(f'    {key}: query {qid} from an earlier attempt could not be used ({type(exc).__name__}); sending a new one')
                qid = None
        if qid is None:
            with QUERY_LIMITER:
                qid = create_query(ws, we)
                _mark_window(key, 'waiting', query_id=qid, query_created_at=datetime.now(timezone.utc).isoformat())
                wait_for_query(qid)
        _mark_window(key, 'draining', query_id=qid)
        n, pages = _drain(qid, key)
        _mark_window(
            key,
            'succeeded',
            query_id=qid,
            rows=n,
            pages=pages,
            files=[_stage_basename(p) for p in list_window_files(STAGING_ABS, key)],
            completed_at=datetime.now(timezone.utc).isoformat(),
        )
        return key, n, 'done'
    except BudgetReached:
        # Keep the query id: the retry re-polls the query instead of queueing a new one.
        _mark_window(key, 'waiting', paused_at=datetime.now(timezone.utc).isoformat())
        raise
    except Exception as e:
        failure = {
            'error': f'{type(e).__name__}: {e}',
            'completed_at': None,
        }
        if qid:
            failure['query_id'] = qid
        _mark_window(key, 'failed', **failure)
        purge_window_files(STAGING_ABS, key, include_partial=True)
        raise


_FATAL_HTTP = {401, 403}


def _http_status(exc):
    status = getattr(getattr(exc, 'response', None), 'status_code', None)
    return status if isinstance(status, int) else None


def is_retryable_window_error(exc) -> bool:
    """Failures a fresh query can fix: failed/cancelled by the service, timeouts, throttling, outages."""
    if isinstance(exc, (AuditQueryFailed, TimeoutError, ConnectionError)):
        return True
    status = _http_status(exc)
    if status is not None:
        return status in _TRANSIENT
    return type(exc).__module__.split('.')[0] in ('requests', 'urllib3')


def _run_window(win):
    """Fetch one window: retry with a new query, then split it when the retries run out.

    Returns (key, rows, outcome, detail). outcome is 'skip', 'done', 'split' (detail = the halves),
    'paused' (TIME_BUDGET_MIN ran out) or 'failed' (detail = the last error). Raises only for 401/403,
    which would fail every window.
    """
    ws, we = win
    key = stable_window_key(ws, we)
    retries = max(int(WINDOW_RETRIES), 0)
    errors, last = [], None
    for attempt in range(1, retries + 2):
        try:
            _, rows, how = _process(win, attempt=attempt)
            return key, rows, how, None
        except BudgetReached:
            return key, 0, 'paused', None
        except Exception as exc:
            if _http_status(exc) in _FATAL_HTTP:
                raise
            last = f'{type(exc).__name__}: {exc}'
            errors.append(f'attempt {attempt}: {last}')
            _mark_window(key, 'failed', attempts=attempt, attempt_errors=errors[-5:])
            if not is_retryable_window_error(exc):
                return key, 0, 'failed', last
            if QUERY_LIMITER.shrink(f'{key}: {last}', halve=_http_status(exc) == 429):
                print(f'    concurrency lowered to {QUERY_LIMITER.limit} after: {last}')
            if isinstance(exc, TimeoutError) and SPLIT_ON_TIMEOUT:
                break
            if budget_exhausted(RUN_DEADLINE):
                return key, 0, 'paused', None
            if attempt <= retries:
                delay = retry_delay(attempt, RETRY_BASE_SEC, RETRY_MAX_SEC)
                print(f'    {key}: {last}; new query in {delay:.0f}s (retry {attempt}/{retries})')
                time.sleep(delay)
    halves = split_window(ws, we, MIN_CHUNK_HOURS)
    if not halves:
        return key, 0, 'failed', last
    _mark_window(
        key,
        'split',
        parts=[stable_window_key(h_ws, h_we) for h_ws, h_we in halves],
        split_at=datetime.now(timezone.utc).isoformat(),
        error=last,
        completed_at=None,
    )
    purge_window_files(STAGING_ABS, key, include_partial=True)
    return key, 0, 'split', halves


def leaf_windows(planned):
    """The windows that hold the data for a plan: split windows are replaced by their parts."""
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    return [leaf for ws, we in planned for leaf in expand_split_windows(snapshot, ws, we, MIN_CHUNK_HOURS)]


def _fmt_range(ws, we):
    return f'{_as_utc_datetime(ws):%Y-%m-%d %H:%M} -> {_as_utc_datetime(we):%Y-%m-%d %H:%M} UTC'


def assert_no_unrecovered_gaps(not_before, before):
    gaps = uncovered_failed_ranges(manifest, not_before, before)
    if gaps:
        lines = '\n'.join(f'  - {_fmt_range(s, e)}' for s, e in gaps)
        raise RuntimeError(
            f'{len(gaps)} earlier audit range(s) failed and were never recovered:\n{lines}\n'
            f'This incremental run starts at {_as_utc_datetime(before):%Y-%m-%d %H:%M} UTC, so it would leave '
            f"them empty. Rerun with MODE='backfill' first: succeeded windows are reused, so only these ranges "
            f'are queried again. To accept the gap instead, set ALLOW_UNRECOVERED_GAPS = True.'
        )


def fetch_windows(planned):
    """Fetch every planned window (retrying and splitting); return (active keys, staged rows)."""
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    fresh_after = trailing_fresh_after(datetime.now(timezone.utc), TRAILING_REUSE_HOURS)
    leaves = leaf_windows(planned)
    pending = [
        (ws, we) for ws, we in leaves
        if not is_window_reusable(
            STAGING_ABS, stable_window_key(ws, we), we, _read_manifest_entry(stable_window_key(ws, we)),
            trailing_cutoff, fresh_after,
        )
    ]
    print(
        f'{len(leaves) - len(pending)} window(s) already reusable; {len(pending)} to fetch '
        f'(<= {QUERY_LIMITER.limit} queries at a time; a failing window is retried {WINDOW_RETRIES}x, '
        f'then split down to {MIN_CHUNK_HOURS:g}h).'
    )
    outcomes, fatal, done, jobs = {}, None, 0, len(pending)
    with ThreadPoolExecutor(max_workers=max(1, int(MAX_CONCURRENT_QUERIES))) as ex:
        running = {ex.submit(_run_window, w): w for w in pending}
        while running:
            finished, _ = wait_futures(list(running), return_when=FIRST_COMPLETED)
            for fut in finished:
                running.pop(fut)
                if fut.cancelled():
                    continue
                try:
                    key, n, how, detail = fut.result()
                except Exception as e:
                    fatal = fatal or e
                    for other in running:
                        other.cancel()
                    continue
                outcomes[key] = how
                done += 1
                if how == 'split':
                    hours = (detail[0][1] - detail[0][0]).total_seconds() / 3600
                    print(f'  [{done}/{jobs}] {key} kept failing -> split into {len(detail)} x {hours:g}h')
                    if fatal is None:
                        for half in detail:
                            running[ex.submit(_run_window, half)] = half
                            jobs += 1
                elif how == 'failed':
                    print(f'  [{done}/{jobs}] {key} FAILED: {detail}')
                elif how == 'paused':
                    print(f'  [{done}/{jobs}] {key} left for the retry (time budget reached)')
                else:
                    print(f'  [{done}/{jobs}] {key} {how} (+{n:,})')
    if fatal is not None:
        raise fatal

    leaves = leaf_windows(planned)
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    keys = [stable_window_key(ws, we) for ws, we in leaves]
    failed = [(w, snapshot.get(k) or {}) for w, k in zip(leaves, keys) if not manifest_succeeded(snapshot.get(k))]
    fetched = [k for k in keys if outcomes.get(k) == 'done']
    retried = [k for k in fetched if int((snapshot.get(k) or {}).get('attempts', 1) or 1) > 1]
    split = sum(1 for how in outcomes.values() if how == 'split')
    rows = sum(int((snapshot.get(k) or {}).get('rows', 0) or 0) for k in keys if manifest_succeeded(snapshot.get(k)))
    shrinks = ', '.join(f'{old}->{new}' for old, new, _ in QUERY_LIMITER.history) or 'unchanged'
    print(
        f'Summary: {len(planned)} planned window(s) -> {len(leaves)} after splits | '
        f'reused {len(leaves) - len(fetched) - len(failed)}, fetched {len(fetched)} ({len(retried)} after a retry), '
        f'split {split}, failed {len(failed)} | concurrency {shrinks} | staged records {rows:,}'
    )
    paused = [k for k, how in outcomes.items() if how == 'paused']
    if paused:
        raise RuntimeError(
            f'Time budget of {TIME_BUDGET_MIN} min reached with {len(paused)} audit window(s) still to fetch '
            f'({len(fetched)} fetched this run). Nothing was written to {OUTPUT_TABLE}. The pipeline retry, or rerunning with the same MODE, carries on from here: '
            f'finished windows are reused and queries still running are picked up again.'
        )
    if failed:
        lines = '\n'.join(
            f'  - {_fmt_range(ws, we)} [{stable_window_key(ws, we)}]: {entry.get("error") or entry.get("status")}'
            for (ws, we), entry in failed
        )
        rerun = "rerun with MODE='backfill'" if MODE == 'backfill' else 'rerun the notebook'
        raise RuntimeError(
            f'{len(failed)} audit window(s) still failed after {WINDOW_RETRIES} retries and splitting down to '
            f'{MIN_CHUNK_HOURS:g}h. Nothing was written to {OUTPUT_TABLE}.\n{lines}\n'
            f'Next step: {rerun}; succeeded windows are reused, so only the windows above are queried again. '
            f'If they keep failing, check for other audit searches running in the tenant and lower '
            f'MAX_CONCURRENT_QUERIES.'
        )
    return set(keys), rows


with MANIFEST_LOCK:
    manifest = _load_manifest()
if MODE == 'incremental' and not ALLOW_UNRECOVERED_GAPS:
    assert_no_unrecovered_gaps(end_date - timedelta(days=max(int(BACKFILL_DAYS), 0)), start_date)
QUERY_LIMITER = AdaptiveLimiter(MAX_CONCURRENT_QUERIES, MIN_CONCURRENT_QUERIES)
THROTTLE_LISTENERS[:] = [lambda: QUERY_LIMITER.shrink('HTTP 429 (throttled) from Graph', halve=True)]
ACTIVE_WINDOW_KEYS, total = fetch_windows(windows)
print(f'Prepared {len(ACTIVE_WINDOW_KEYS)} active window(s); staged files will be read only from this run scope. Manifest: {MANIFEST}')

## 6. Parse the staged records (Spark)

Each audit record is parsed with Spark built-in functions — no Python UDFs — following the M365 Usage Analytics
processor (v2.6.1): non-human identities are dropped, operations are canonicalised (`FileViewed` → `FileAccessed`,
`MeetingParticipantJoined` → `MeetingParticipantDetail`, `ConnectedAIAppInteraction` → `AIAppInteraction`; the original
name is kept in `SourceOperation`), each AI-app record counts as its exploded row count (largest Copilot array, capped at 1,000), and each `MailItemsAccessed`
event counts the folder items it touched.

In [ ]:
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, BooleanType, StringType, StructField, StructType

ROLLUP_HEADER = ['UserId', 'CreationDate', 'Operation', 'Workload', 'SourceFileExtension', 'AppHost',
                 'EventCount', 'ItemsAccessedCount', 'CreationTime', 'MaxCreationTime',
                 'AgentId', 'AgentName', 'ContextType', 'IsAgentInteraction']
AI_APP_HEADER = ['Id', 'UserId', 'CreationDate', 'CreationTime', 'Operation', 'Workload', 'AppHost', 'AgentId',
                 'AgentName', 'ContextType', 'ThreadId', 'PromptCount', 'ResponseCount', 'EventCount', 'IsAgent']

OP_RENAME = {'FileViewed': 'FileAccessed', 'MeetingParticipantJoined': 'MeetingParticipantDetail',
             'ConnectedAIAppInteraction': 'AIAppInteraction'}
AGENT_APPHOSTS = ['agent', 'copilotstudio', 'declarativeagent', 'customengineagent']
EVENT_ROW_CAP = 1000

_S = StringType()
STAGE_SCHEMA = StructType([StructField(c, _S) for c in (
    'id', 'createdDateTime', 'operation', 'userPrincipalName', 'userId', 'auditLogRecordType', 'auditData')])
AUDIT_SCHEMA = StructType(
    [StructField(c, _S) for c in ('UserId', 'Operation', 'Workload', 'SourceFileExtension', 'AppHost',
                                  'AgentId', 'AgentName', 'CreationTime', 'CopilotEventData')]
    + [StructField('Folders', ArrayType(StructType([StructField('FolderItems', ArrayType(_S))]))),
       StructField('_corrupt_record', _S)])
COPILOT_SCHEMA = StructType(
    [StructField(c, _S) for c in ('AppHost', 'AgentId', 'AgentName', 'ThreadId')]
    + [StructField('Messages', ArrayType(StructType([StructField('isPrompt', BooleanType())]))),
       StructField('Contexts', ArrayType(StructType([StructField('Type', _S), StructField('Items', ArrayType(_S))])))]
    + [StructField(c, ArrayType(_S)) for c in ('AccessedResources', 'SensitivityLabels', 'AISystemPlugin',
                                              'ModelTransparencyDetails')])


def _norm(c: Column) -> Column:
    """Trimmed text; null, 'null' and 'none' become ''."""
    c = F.regexp_replace(F.coalesce(c, F.lit('')), r'(?U)^\s+|\s+$', '')
    return F.when(F.lower(c).isin('null', 'none'), F.lit('')).otherwise(c)


def _first(*cols: Column) -> Column:
    """First value that is neither null nor '' (Python `a or b`)."""
    return F.coalesce(*[F.when(c != '', c) for c in cols], F.lit(''))


def _size(c: Column) -> Column:
    return F.greatest(F.size(c), F.lit(0))


def _is_human_upn(u: Column) -> Column:
    local, domain = F.substring_index(u, '@', 1), F.substring_index(u, '@', -1)
    return (~u.rlike(r'(?i)^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')
            & ~F.lower(u).startswith('supervisoryreview{')
            & ((F.length(u) - F.length(F.regexp_replace(u, '@', ''))) == 1)
            & local.rlike(r'(?U)^[^\s\\@]+$')
            & domain.contains('.') & ~domain.startswith('.') & ~domain.endswith('.'))


def _creation_date(c: Column) -> Column:
    """Any Purview date format -> 'YYYY-MM-DDT00:00:00.000Z'."""
    raw = _norm(c)
    ts = F.coalesce(*[F.call_function('try_to_timestamp', raw, F.lit(fmt))
                      for fmt in ('M/d/yyyy h:mm:ss a', 'M/d/yyyy H:mm:ss', 'M/d/yyyy')])
    return (F.when(raw == '', F.lit(''))
             .when((F.length(raw) >= 10) & (F.substring(raw, 5, 1) == '-'),
                   F.concat(F.substring(raw, 1, 10), F.lit('T00:00:00.000Z')))
             .when(ts.isNotNull(), F.concat(F.date_format(ts, 'yyyy-MM-dd'), F.lit('T00:00:00.000Z')))
             .otherwise(raw))


def _creation_time(c: Column) -> Column:
    """Purview timestamp -> 'YYYY-MM-DDTHH:MM:SS.mmmZ'; anything unrecognised is kept as-is."""
    raw = _norm(c)
    iso = r'^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})(\.\d{1,6})?Z?$'
    base, frac = F.regexp_extract(raw, iso, 1), F.regexp_extract(raw, iso, 2)
    iso_ok = (base != '') & F.call_function('try_to_timestamp', base, F.lit("yyyy-MM-dd'T'HH:mm:ss")).isNotNull()
    ts = F.coalesce(*[F.call_function('try_to_timestamp', raw, F.lit(fmt))
                      for fmt in ('M/d/yyyy h:mm:ss a', 'M/d/yyyy H:mm:ss')])
    return (F.when(iso_ok, F.concat(base, F.lit('.'), F.substring(F.rpad(F.substring(frac, 2, 6), 3, '0'), 1, 3), F.lit('Z')))
             .when(ts.isNotNull(), F.date_format(ts, "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
             .otherwise(raw))


def parse_audit(raw: DataFrame) -> DataFrame:
    """Staged Graph audit records -> one row per kept record with the rollup keys and counts."""
    ad = F.from_json('auditData', AUDIT_SCHEMA)
    # Only JSON objects are audit records; malformed text or arrays are dropped as the processor does.
    df = (raw.withColumn('_ad', ad)
             .where(F.col('auditData').rlike(r'^\s*\{') & F.col('_ad').isNotNull() & F.col('_ad._corrupt_record').isNull())
             .withColumn('_cedraw', F.col('_ad.CopilotEventData'))
             .withColumn('_hasced', F.coalesce(F.col('_cedraw').rlike(r'^\s*\{\s*"'), F.lit(False)))
             .withColumn('_ced', F.when(F.col('_hasced'), F.from_json('_cedraw', COPILOT_SCHEMA))))
    a, c = F.col('_ad'), F.col('_ced')
    op = _norm(_first(a['Operation'], F.col('operation')))
    rename = F.create_map(*[F.lit(x) for kv in OP_RENAME.items() for x in kv])
    df = (df.withColumn('UserId', _norm(_first(a['UserId'], F.col('userPrincipalName'), F.col('userId'))))
            .where(_is_human_upn(F.col('UserId')))
            .withColumn('SourceOperation', op)
            .withColumn('Operation', F.coalesce(F.element_at(rename, op), op)))
    ctx_items = F.coalesce(F.array_max(F.transform(c['Contexts'], lambda x: _size(x['Items']))), F.lit(0))
    copilot_rows = F.greatest(
        F.lit(1), *[_size(c[k]) for k in ('Messages', 'Contexts', 'AccessedResources', 'SensitivityLabels',
                                          'AISystemPlugin', 'ModelTransparencyDetails')],
        F.when(F.col('Operation') == 'CopilotInteraction', ctx_items).otherwise(0))
    folder_items = F.aggregate(a['Folders'], F.lit(0),
                               lambda acc, f: acc + _size(f['FolderItems']))
    hc = F.col('_hasced')
    df = (df.withColumn('UserKey', F.lower('UserId'))
            .withColumn('CreationDate', _creation_date(F.col('createdDateTime')))
            .withColumn('Workload', _norm(a['Workload']))
            .withColumn('SourceFileExtension', F.lower(_norm(a['SourceFileExtension'])))
            .withColumn('AppHost', _norm(_first(c['AppHost'], a['AppHost'])))
            .withColumn('AgentId', _norm(_first(c['AgentId'], a['AgentId'])))
            .withColumn('AgentName', _norm(_first(c['AgentName'], a['AgentName'])))
            .withColumn('ContextType', _norm(F.element_at(c['Contexts'], 1)['Type']))
            .withColumn('CreationTime', _creation_time(a['CreationTime']))
            .withColumn('EventCount', F.when(hc, F.least(copilot_rows, F.lit(EVENT_ROW_CAP))).otherwise(1).cast('long'))
            .withColumn('ItemsAccessedCount', F.when(F.col('Operation') != 'MailItemsAccessed', 0)
                                               .when(folder_items > 0, folder_items).otherwise(1).cast('long'))
            .withColumn('IsAgent', (F.col('AgentId') != '') | F.lower('AppHost').isin(AGENT_APPHOSTS)
                                   | (F.col('Operation') == 'AIAppInteraction'))
            .withColumn('HasCopilotData', hc)
            .withColumn('ThreadId', F.when(hc, _norm(c['ThreadId'])).otherwise(''))
            .withColumn('PromptCount', F.when(hc, _size(F.filter(c['Messages'], lambda m: m['isPrompt']))).otherwise(0).cast('long'))
            .withColumn('ResponseCount', F.when(hc, _size(F.filter(c['Messages'], lambda m: ~m['isPrompt']))).otherwise(0).cast('long')))
    return df.select('id', 'UserKey', 'UserId', 'CreationDate', 'SourceOperation', 'Operation', 'Workload', 'SourceFileExtension', 'AppHost',
                     'AgentId', 'AgentName', 'ContextType', 'CreationTime', 'EventCount', 'ItemsAccessedCount',
                     'IsAgent', 'HasCopilotData', 'ThreadId', 'PromptCount', 'ResponseCount')

## 7. Read and parse the staged records (distributed)

Records that appear in more than one window are dropped by their audit record `id` before parsing.

In [ ]:
_files = list_active_stage_files(STAGING_ABS, ACTIVE_WINDOW_KEYS)
if _files:
    _raw = spark.read.schema(STAGE_SCHEMA).json(_files)
else:
    _raw = spark.createDataFrame([], STAGE_SCHEMA)
_raw = _raw.withColumn('_key', F.coalesce('id', F.sha2(F.concat_ws('|', 'createdDateTime', 'operation', 'auditData'), 256)))
recs = parse_audit(_raw.dropDuplicates(['_key'])).persist()
_parsed = recs.count()
print(f'Parsed rows: {_parsed:,} (from {len(_files)} active window file(s))')
if _parsed == 0 and _files:
    _d = _raw.agg(F.count(F.lit(1)).alias('raw'),
                  F.sum(F.col('auditData').isNull().cast('int')).alias('no_auditData'),
                  F.sum(F.col('operation').isNull().cast('int')).alias('no_operation')).collect()[0]
    print(f"Staged rows: {_d['raw']:,}; missing auditData: {_d['no_auditData']:,}; missing operation: {_d['no_operation']:,}")
    if _d['raw'] and _d['no_auditData'] == _d['raw']:
        raise RuntimeError(
            f'The staged files under {STAGING_DIR} are not raw Graph audit records (no auditData). They were probably '
            'written by an older ingester version. Delete that folder and rerun with RUN_MODE backfill.')

## 8. Roll up per user and day

Same grouping as the processor's `run_rollup`: the *Rollup* key is user, day, operation, workload, file extension, app host,
agent and context type. AI-app interactions are kept one row per audit record in their own tables; the template rolls them
up together with `Copilot_Interactions_Parsed` and derives *Session Stats* from them.

In [ ]:
_GROUP = ['UserKey', 'CreationDate', 'Operation', 'Workload', 'SourceFileExtension', 'AppHost',
          'AgentId', 'AgentName', 'ContextType']
_ct = F.when(F.col('CreationTime') != '', F.col('CreationTime'))
rollup = (recs.where(~F.col('Operation').isin('CopilotInteraction', 'AIAppInteraction')).groupBy(*_GROUP).agg(
            F.min('UserId').alias('UserId'),
            F.sum('EventCount').alias('EventCount'),
            F.sum('ItemsAccessedCount').alias('ItemsAccessedCount'),
            F.coalesce(F.min(_ct), F.lit('')).alias('CreationTime'),
            F.coalesce(F.max(_ct), F.lit('')).alias('MaxCreationTime'),
            F.max(F.col('IsAgent').cast('int')).alias('_agent'))
          .withColumn('IsAgentInteraction', F.when(F.col('_agent') == 1, 'TRUE').otherwise('FALSE'))
          .select(*ROLLUP_HEADER))

def ai_app_records(source_operation):
    return (recs.where(F.col('SourceOperation') == source_operation)
            .withColumn('Id', F.col('id'))
            .withColumn('Operation', F.col('SourceOperation'))
            .withColumn('IsAgent', F.when(F.col('IsAgent'), 'TRUE').otherwise('FALSE'))
            .select(*AI_APP_HEADER))

connected_ai = ai_app_records('ConnectedAIAppInteraction')
ai_app = ai_app_records('AIAppInteraction')

## 9. Write to Lakehouse Delta tables

Backfill overwrites all three tables. Incremental re-reads whole days from `REPLACE_FROM`, so it deletes those days and appends the fresh rollup — re-running is safe.

In [ ]:
def write_days(df, table):
    if WRITE_MODE == 'replace_dates' and spark.catalog.tableExists(table):
        from delta.tables import DeltaTable
        DeltaTable.forName(spark, table).delete(F.col('CreationDate') >= F.lit(REPLACE_FROM))
        df.write.format('delta').mode('append').saveAsTable(table)
        return f'replaced days from {REPLACE_FROM[:10]}'
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table)
    return 'overwrite'

for _df, _table in ((rollup, OUTPUT_TABLE), (connected_ai, CONNECTED_AI_TABLE), (ai_app, AI_APP_TABLE)):
    _how = write_days(_df, _table)
    print(f'✓ {_table}: {spark.table(_table).count():,} rows ({_how})')
recs.unpersist()

## 10. Verify

Event counts by workload and operation for the last day loaded.

In [ ]:
_last = spark.table(OUTPUT_TABLE).agg(F.max('CreationDate')).collect()[0][0]
(spark.table(OUTPUT_TABLE).where(F.col('CreationDate') == _last)
    .groupBy('Workload', 'Operation').agg(F.sum('EventCount').alias('Events'), F.sum('ItemsAccessedCount').alias('ItemsAccessed'))
    .orderBy(F.desc('Events')).show(30, truncate=False))